<a href="https://colab.research.google.com/github/wtryab-re/data-preprocessing/blob/main/Loan_Approval_Dataset_EDA_%26_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install and Import

In [486]:
!pip install -q pandas numpy seaborn matplotlib imblearn scikit-learn fg_data_profiling

In [487]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from data_profiling import ProfileReport

#Helps impute values faster
from sklearn.impute import SimpleImputer

In [488]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', "{:.3f}".format)
sns.set_style("darkgrid")

#Dataframe

In [489]:
df = pd.read_csv("loan.csv")
df.shape

(614, 13)

In [490]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.000,NaN,360.000,1.000,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.000,128.000,360.000,1.000,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.000,66.000,360.000,1.000,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.000,120.000,360.000,1.000,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.000,141.000,360.000,1.000,Urban,Y


#Data EDA

In [ ]:
EDA_Report = ProfileReport(df, title="Loan Approval Profiling Report", )
EDA_Report.to_file("Loan_Approval_Profiling_Report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 35.46it/s]


#Data Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

#DO NOT WORK ON THE ACTUAL DATASET

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy.shape

#Basic Cleanup & Data Quality Checks

In [ ]:
df_copy.columns = [col.lower() for col in df_copy.columns]
df_copy.columns = [col.replace("_", "") for col in df_copy.columns]
df_copy.columns

In [ ]:
df.nunique()

ID-like, Constant and Quasi Constant Columns

In [ ]:
id_like_cols = []
constant_cols = []
quasi_constant_cols = []

#constant cols - nunique = 1
#quasiconstant - value_counts max proportion > 90
#id-like - nunique/rows*100 >90

for col in df_copy.columns:
  if df_copy[col].nunique() == 1:
    constant_cols.append(col)
  elif df_copy[col].nunique()/df_copy.shape[0]*100 > 90:
    id_like_cols.append(col)
  elif df_copy[col].value_counts(normalize=True).max()*100 > 90:
    quasi_constant_cols.append(col)

print(f"ID-Like Columns: {id_like_cols}")
print(f"Constant Columns: {constant_cols}")
print(f"Quasi Constant Columns: {quasi_constant_cols}")

drop_cols = id_like_cols + constant_cols + quasi_constant_cols

In [ ]:
df_copy.drop(drop_cols, inplace = True, axis = 1)
df_copy.shape

##Missing Values Check

In [ ]:
df_copy.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

##String Inconsistencies

In [ ]:
text_cols = df_copy.select_dtypes(exclude=np.number).columns
df_copy[text_cols] = df_copy[text_cols].apply(lambda x: x.str.strip())
df_copy[text_cols] = df_copy[text_cols].apply(lambda x: x.str.lower())

##High Null Columns

In [ ]:
missing_count = df_copy.isna().sum().sort_values(ascending=False)
missing_percentage = (df_copy.isna().mean()*100).sort_values(ascending=False)

missing_df = pd.DataFrame({"missing_count":missing_count,"missing_percentage":missing_percentage})
missing_df[missing_df["missing_count"]>0]

##High Zero Columns

In [ ]:
numeric_columns = df_copy.select_dtypes(include=np.number).columns
high_zero_threshold = 40
zero_values = (df_copy[numeric_columns]==0).mean()*100
zero_values.sort_values(ascending=False)

##Check for Missing Target Values

In [ ]:
target = "loanstatus"

In [ ]:
df_copy[target].isna().sum()

##Drop Missing Target Value Data

In [ ]:
df_copy.dropna(subset=target)
df_copy[target].isna().sum()

#Train Test Split

In [ ]:
X = df_copy.drop(target, axis=1)
y = df_copy[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

#Separate Num Cols and Categorical Cols

In [ ]:
X_train_processed.head()

In [ ]:
X_train_processed.columns

In [ ]:
X_train_processed.nunique()

In [ ]:
for col in X_train_processed.columns:
  print(col, X_train_processed[col].unique())

In [ ]:
len(X_train_processed.columns)

In [ ]:

  # gender - cat unorder
  # married - cat unorder
  # dependents - cat ordinal
  # education - cat unorder
  # credit_history - cat unorder
  # selfemployed - cat unorder
  # propertyarea - cat unorder binary no encoding needed

  # applicantincome = number continuous
  # coapplicantincome = number continuous
  # loanamount - number continuous
  # loanamountterm - number discrete

  #loanstatus - category binary encode

In [ ]:
preprocessingaction = {
    "columns": ["gender", "married", "dependents", "education", "credithistory", "selfemployed", "propertyarea", "loanstatus (target)", "applicantincome", "coapplicantincome", "loanamount", "loanamountterm"],
    "datatype": ["category", "category", "category", "category", "category", "category", "category", "category", "number", "number", "number", "number"],
    "type": ["binary", "binary", "ordinal", "binary", "binary", "binary", "nominal", "binary" ,"continuous", "continuous","continuous", "discrete"],
    "action": ["label", "label", "label", "label", "already encoded", "label", "one hot", "label", "scale", "scale", "scale", "scale"]
    }


preprocessing_info = pd.DataFrame(preprocessingaction)
preprocessing_info

In [ ]:
num_cols = ["applicantincome", "coapplicantincome", "loanamount", "loanamountterm"]
cat_cols = ["gender", "married", "dependents", "education", "credithistory", "selfemployed", "propertyarea"]

In [ ]:
num_cols  = X_train_processed.select_dtypes(include=np.number).columns.to_list()
cat_cols = X_train_processed.select_dtypes(exclude=np.number).columns.to_list()

num_cols.remove("credithistory")
cat_cols.append("credithistory")

print("Numerical Columns", num_cols)
print("Categorical Columns", cat_cols)

#Handling Missing Values

In [ ]:
X_train_handled = X_train_processed.copy()
X_test_handled = X_test_processed.copy()

##Categorical Columns

Impute using MOST FREQUENT - Mode

In [ ]:
#Before
X_train_handled[cat_cols].isna().sum()

In [ ]:
cat_imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
X_train_handled[cat_cols] = cat_imputer.fit_transform(X_train_handled[cat_cols])
X_test_handled[cat_cols] = cat_imputer.transform(X_test_handled[cat_cols])

In [ ]:
#after
X_train_handled[cat_cols].isna().sum()

#Numerical Columns

In [ ]:
#before
X_train_handled[num_cols].isna().sum()

In [ ]:
num_imputer = SimpleImputer(strategy="median")

In [ ]:
X_train_handled[num_cols]= num_imputer.fit_transform(X_train_handled[num_cols])
X_test_handled[num_cols] = num_imputer.transform(X_test_handled[num_cols])

In [ ]:
#after
X_train_handled[num_cols].isna().sum()

#Outlier Detection

In [ ]:
X_train_outlier_handled = X_train_handled.copy()
X_test_outlier_handled = X_test_handled.copy()

##1. Boxplots

In [ ]:
for col in num_cols:
  sns.boxplot(x = X_train_outlier_handled[col])
  plt.title(col)
  plt.show()
  print()

In [ ]:
#imo no need to touch loanamountterm it is discrete
#rest can be touched
#yay i was right
num_cols

In [ ]:
desc = {
    "column": num_cols,
    "cap outliers": [1,1,1,0],
    "reason": ["hella outliers","hella outliers","hella outliers", "discrete so no"]
}

pd.DataFrame(desc)


In [ ]:
outlier_columns = list( filter(lambda x : x not in ["loanamountterm"], num_cols))
outlier_columns

###ALWAYS CAP NUMERICAL FEATURES

###DO NOT CAP DISCRETE OR BINARY COLUMNS

###ALWAYS HANDLE MISSING VALUES BEFORE OUTLIERS


##IQR METHOD IS USED FOR NON-NORMAL DISTRIBUTION DATA

##Z-SCORE FOR NORMAL only

#Outlier Handling

In [ ]:
def boundary_finder(df, col):
  Q3 = df[col].quantile(0.75)
  Q1 = df[col].quantile(0.25)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5*IQR
  upper_bound = Q3 + 1.5*IQR

  return lower_bound, upper_bound

In [ ]:
def iqr_method():
  train_boundaries={}
  for col in outlier_columns:
    lower_bound, upper_bound = boundary_finder(X_train_outlier_handled, col)

    X_train_outlier_handled.loc[X_train_outlier_handled[col]<lower_bound, col] = lower_bound
    X_train_outlier_handled.loc[X_train_outlier_handled[col]>upper_bound, col] = upper_bound

    X_test_outlier_handled.loc[X_test_outlier_handled[col]<lower_bound, col] = lower_bound
    X_test_outlier_handled.loc[X_test_outlier_handled[col]>upper_bound, col] = upper_bound

    train_boundaries[col] = [lower_bound, upper_bound]

  return train_boundaries

In [ ]:
train_boundaries = iqr_method()
train_boundaries

In [ ]:
for col in outlier_columns:
  sns.boxplot(x = X_train_outlier_handled[col])
  plt.show()
  print()

#Encoding Categorical Columns

In [ ]:
X_train_encoded = X_train_outlier_handled.copy()
X_test_encoded = X_test_outlier_handled.copy()

In [ ]:
preprocessing_info

##Cat cols grouping

In [ ]:
label_columns = preprocessing_info[preprocessing_info["action"]=="label"]["columns"].to_list()
label_columns.remove("loanstatus (target)")
label_columns

In [ ]:
one_hot_columns = preprocessing_info[preprocessing_info["action"]=="one hot"]["columns"].to_list()
one_hot_columns

##Label Encodings

###1. Encode the target - loanstatus

In [ ]:
encoders = {}

In [ ]:
le = LabelEncoder()

In [ ]:
encoded_y_train = pd.Series(le.fit_transform(y_train), name= target)
encoded_y_test = pd.Series(le.transform(y_test), name=target)
encoders["target"] = le

In [ ]:
for col in label_columns:
  X_train_encoded[col] = le.fit_transform(X_train_encoded[col])
  X_test_encoded[col] = le.transform(X_test_encoded[col])
  encoders[col] = le

X_train_encoded.tail()

In [ ]:
encoders

##One Hot Encoding

In [ ]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

In [ ]:
ohe.fit(X_train_encoded[one_hot_columns])
X_train_ohe_encoded = ohe.transform(X_train_encoded[one_hot_columns])
X_test_ohe_encoded= ohe.transform(X_test_encoded[one_hot_columns])

In [ ]:
ohe_col_names = ohe.get_feature_names_out(one_hot_columns)
ohe_col_names

In [ ]:
X_train_ohe_encoded = pd.DataFrame(X_train_ohe_encoded, columns=ohe_col_names, index=X_train_encoded.index)
X_test_ohe_encoded = pd.DataFrame(X_test_ohe_encoded, columns=ohe_col_names, index=X_test_encoded.index)

In [ ]:
X_train_fully_encoded = pd.concat([X_train_encoded.drop(one_hot_columns, axis=1), X_train_ohe_encoded], axis=1)
X_test_fully_encoded = pd.concat([X_test_encoded.drop(one_hot_columns, axis=1), X_test_ohe_encoded], axis=1)
X_train_fully_encoded.head()

#Handling Imbalanced Data

Only to be done on training data

In [ ]:
encoded_y_train.value_counts()

In [ ]:
encoded_y_train.value_counts(normalize = True)

##Oversampling will be used here because dataset small. Can use SMOTE or RandomOversampler

Using SMOTE IN THIS CASE because continuous features.

In [ ]:
X_train_resampled, y_train_resampled = SMOTE(random_state=42).fit_resample(X_train_fully_encoded, encoded_y_train)
X_train_resampled.shape, y_train_resampled.shape

In [ ]:
encoded_y_train

#Standardization & Normalization

In [ ]:
X_train_standard = X_train_resampled.copy()
X_test_standard = X_test_fully_encoded.copy()

y_train_final = y_train_resampled.copy()
y_test_final = encoded_y_test.copy()

X_train_standard.shape, X_test_standard.shape, y_train_final.shape, y_test_final.shape

In [ ]:
num_cols

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_standard)
X_train_final = pd.DataFrame(scaler.transform(X_train_standard), columns=X_train_standard.columns)
X_test_final = pd.DataFrame(scaler.transform(X_test_standard), columns=X_test_standard.columns)

X_train_final.head()

In [ ]:
df_train_processed = pd.concat([X_train_final, y_train_final], axis=1)
df_test_processed = pd.concat([X_test_final, y_test_final], axis=1)

df_train_processed.columns =list( X_train_final.columns )+ [target]
df_test_processed.columns =list( X_train_final.columns )+ [target]

In [ ]:
df_train_processed.head()

In [ ]:
df_test_processed.head()

#SAVE

In [ ]:
df_train_processed.to_csv("df_train_processed.csv", index=False)
df_test_processed.to_csv("df_test_processed.csv", index=False)

In [ ]:
#jeez this took FOREVER